# Plate 02 · Synthetic Egyptian Plate Generator

Real Egyptian plates are scarce and unbalanced (few taxi / transport / night images).
This generator renders **100k+ labelled plates** with the real layout: top band (colour = plate type)
with "مصر EGYPT", Arabic letters on the right half, Arabic-Indic digits on the left half.

Fonts: download **Noto Kufi Arabic** / **Noto Naskh Arabic** (SIL OFL) from Google Fonts into `Data/Plates/Fonts/`.
Augmentations: perspective, motion blur, night (low gamma + noise), rain streaks, glare, dirt, JPEG artefacts.

In [ ]:
# --- Setup: make Common_Utils importable (works locally, on Kaggle and on Colab) ---
import sys
from pathlib import Path

# On Kaggle/Colab first clone the repo, e.g.:
# !git clone https://github.com/AhmedADesoky/Car_Services_App.git && %cd Car_Services_App/AI_Models/Plate_Recognition
AI_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(AI_ROOT))

from Common_Utils.Paths import DATA, RUNS, WEIGHTS, model_dir
from Common_Utils.Data_Utils import seed_everything
seed_everything(42)
print("Data:", DATA, "| Runs:", RUNS, "| Weights:", WEIGHTS)

In [ ]:
import random
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from Common_Utils.Plate_Normalizer import PLATE_LETTERS, ARABIC_INDIC_DIGITS

FONTS = sorted((DATA / "Plates" / "Fonts").glob("*.ttf"))
OUT = DATA / "Plates" / "Synthetic"
OUT.mkdir(parents=True, exist_ok=True)
assert FONTS, "Put at least one Arabic .ttf font in Data/Plates/Fonts/"

BANDS = {  # plate type -> top band colour (RGB)
    "private": (142, 195, 234), "taxi": (242, 148, 58), "transport": (214, 48, 49),
    "commercial": (238, 214, 84), "other": (120, 120, 120),
}

In [ ]:
def render_plate(letters: str, digits: str, kind: str, font_path: Path, w=400, h=200) -> Image.Image:
    plate = Image.new("RGB", (w, h), (250, 250, 250))
    d = ImageDraw.Draw(plate)
    band_h = int(h * 0.28)
    d.rectangle([0, 0, w, band_h], fill=BANDS[kind])
    small = ImageFont.truetype(str(font_path), int(band_h * 0.7))
    d.text((w * 0.08, band_h * 0.1), "EGYPT", font=small, fill=(10, 10, 10))
    d.text((w * 0.70, band_h * 0.0), "مصر", font=small, fill=(10, 10, 10))
    big = ImageFont.truetype(str(font_path), int(h * 0.48))
    d.line([(w / 2, band_h), (w / 2, h)], fill=(20, 20, 20), width=3)          # middle divider
    d.text((w * 0.55, band_h + h * 0.05), " ".join(letters), font=big, fill=(10, 10, 10))  # right half
    d.text((w * 0.05, band_h + h * 0.05), " ".join(digits), font=big, fill=(10, 10, 10))   # left half
    d.rectangle([1, 1, w - 2, h - 2], outline=(20, 20, 20), width=4)
    return plate

def random_text():
    letters = "".join(random.choices(PLATE_LETTERS, k=random.randint(2, 3)))
    digits = "".join(random.choices(ARABIC_INDIC_DIGITS, k=random.randint(3, 4)))
    return letters, digits

l, n = random_text()
render_plate(l, n, "private", FONTS[0]).resize((200, 100))

In [ ]:
import albumentations as A

augment = A.Compose([
    A.Perspective(scale=(0.02, 0.08), p=0.8),
    A.OneOf([A.MotionBlur(blur_limit=7), A.GaussianBlur(blur_limit=(3, 5)), A.Defocus(radius=(1, 3))], p=0.6),
    A.OneOf([A.RandomGamma(gamma_limit=(40, 80)), A.RandomBrightnessContrast(-0.4, 0.3)], p=0.6),  # night / dusk
    A.RandomRain(p=0.15), A.RandomSunFlare(src_radius=60, p=0.08),
    A.GaussNoise(p=0.4), A.ImageCompression(quality_range=(30, 90), p=0.5),
])

def make_dataset(count=100_000, start=0):
    rows = []
    for i in range(start, start + count):
        letters, digits = random_text()
        kind = random.choices(list(BANDS), weights=[60, 15, 10, 10, 5])[0]
        img = np.array(render_plate(letters, digits, kind, random.choice(FONTS)))
        img = augment(image=img)["image"]
        img = Image.fromarray(img).resize((int(48 * 2), 48))
        name = f"syn_{i:07d}.png"
        img.save(OUT / name)
        rows.append({"file": name, "text": f"{' '.join(letters)} {digits}", "letters": letters, "digits": digits, "kind": kind})
    return pd.DataFrame(rows)

labels = make_dataset(count=2_000)   # raise to 100_000+ on GPU machines
labels.to_csv(OUT / "labels.csv", index=False)
labels.head()